# Compile post-AGB binary systems

Combines two sources:

### 1. Kluska et al. 2022
https://ui.adsabs.harvard.edu/abs/2022A%26A...658A..36K/abstract

_A population of transition disks around evolved stars: Fingerprints of planets._  
_Catalog of disks surrounding Galactic post-AGB binaries._  
Kluska, J.; Van Winckel, H.; Coppée, Q.; Oomen, G.-M.; Dsilva, K.; Kamath, D.; Bujarrabal, V.; Min, M.

Output: `Kluska2021_postAGB.raw.json` (85 systems)

### 2. Oomen et al. 2018
https://ui.adsabs.harvard.edu/abs/2018A%26A...620A..85O/abstract

_Post-AGB stars with hot dust and binarity as a tool for binary stellar evolution._

Output: `Oomen2018_postAGB.raw.json` (33 systems)

In [18]:
import numpy as np
import pandas as pd
import json
import re
import subprocess
import ast
import os
import sys
from pathlib import Path
import math

# Ensure project root is on sys.path
proj_root = Path('/Users/liekevanson/Documents/Projects/post_mt_review').resolve()
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

from paths import DATA_DIR, RAW_JSON_DIR

---
## 1. Kluska et al. 2022

Parses coordinates, period, and eccentricity directly from the plain-text table file.

In [19]:
KLUSKA_INPUT_PREVIEW = DATA_DIR / "from_others" / "Kluska_2021_table1.txt"
kluska_table_full = pd.read_csv(KLUSKA_INPUT_PREVIEW, sep=",", skiprows=[1])  # Quick check of the raw data format
kluska_table_full.columns = kluska_table_full.columns.str.strip()

display(kluska_table_full)
display(kluska_table_full[kluska_table_full['IRAS'].astype(str).str.contains('IRAS06054', na=False)])


,IRAS,Name,RA,DE,Cat,H-Ks(e),W1-W3(e),Teff,LIR,Porb,ecc,yes/no,[Fe/H],[Zn/Ti],TTurnOff,Ref
0,IRAS01427+4633,[BD+46.442],01 45 47.03,+46 49 00.97,Cat. 1,0.46 0.07,2.83 0.05,6250,16,140.82,0.0,no,-0.7,-0.2,,h
1,IRAS04166+5719,[TW Cam],04 20 48.1,+57 26 26.,Cat. 1,0.57 0.05,3.01 0.08,4800,46,662.2,0.25,yes,-0.5,0.3,1100,a
2,IRAS04440+2605,[RV Tau],04 47 06.8,+26 10 44.,Cat. 1,0.61 0.03,3.67 0.09,4500,39,1198.0,,yes,-0.4,0.5,1100,a
3,IRAS06160-1701,[UY CMa],06 18 16.367,-17 02 34.72,Cat. 1,0.59 0.06,3.84 0.04,5500,69,,,no,0.0,1.8,1000,a
4,,[EZ Gem],06 46 04.47,+13 05 02.4,Cat. 1,0.77,2.43 0.03,6555,53,,,no,0.0,,,b
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,IRAS19125+0343,[BD+03 3950],19 15 01.1809,+03 48 42.703,Cat. 0,1.29 0.04,3.38 0.14,7750,133,519.7,0.24,yes,-0.5,2.3,1400,a
81,IRAS19472+4254,[DF Cyg],19 48 53.9423,+43 02 14.529,Cat. 0,1.16 0.02,2.9 0.04,4800,51,784.,,yes,0.0,-0.7,,b
82,IRAS19548+1951,[RS Sge],19 57 06.38,+19 59 43.7,Cat. 0,1.35 0.04,2.85 0.05,6000,308,,,yes,0.0,,,b
83,IRAS20056+1834,[QY Sge],20 07 54.8,+18 42 57.,Cat. 0,1.36 0.04,4.43 0.07,5850,574,,,no,-0.4,1.2,1500,a


,IRAS,Name,RA,DE,Cat,H-Ks(e),W1-W3(e),Teff,LIR,Porb,ecc,yes/no,[Fe/H],[Zn/Ti],TTurnOff,Ref
66,IRAS06054+2237,[SS Gem],06 08 35.1084,+22 37 01.937,Cat. 4,0.12 0.04,0.83 0.05,5400,1,,,no,-1.0,2.0,1100,b


In [20]:
# ---------- coordinate parsing ----------
def _split_sexagesimal(s):
    """Split on whitespace/commas/colons; returns list of 3 floats or None."""
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return None
    parts = str(s).strip().replace(":", " ").replace(",", " ").split()
    if len(parts) != 3:
        return None
    try:
        return [float(p) for p in parts]
    except ValueError:
        return None

def hms_to_deg(hms):
    parts = _split_sexagesimal(hms)
    if parts is None:
        return None
    h, m, s = parts
    return 15.0 * (h + m / 60.0 + s / 3600.0)

def dms_to_deg(dms):
    if dms is None:
        return None
    s = str(dms).strip()
    sign = -1.0 if s.lstrip().startswith("-") else 1.0
    parts = _split_sexagesimal(s)
    if parts is None:
        return None
    d, m, sec = parts
    return sign * (abs(d) + m / 60.0 + sec / 3600.0)

# ---------- value/error helpers ----------
def _clean(x):
    """Return None for NaN / empty strings, else stripped string."""
    if x is None:
        return None
    if isinstance(x, float) and math.isnan(x):
        return None
    s = str(x).strip()
    return s if s else None

def _to_float(x):
    s = _clean(x)
    if s is None:
        return None
    s = s.replace(",", "")
    try:
        return float(s)
    except ValueError:
        return None

def _parse_value_err(cell):
    """
    Parse a 'value error' cell like '0.46 0.07' or '0.77' or ''.
    Returns (value, err) with err possibly None.
    """
    s = _clean(cell)
    if s is None:
        return (None, None)
    tokens = s.split()
    try:
        val = float(tokens[0])
    except (ValueError, IndexError):
        return (None, None)
    err = None
    if len(tokens) > 1:
        try:
            err = float(tokens[1])
        except ValueError:
            err = None
    return (val, err)

def _triplet(val, err=None):
    """Schema triplet [err-, value, err+]. Symmetric error if scalar."""
    if val is None:
        return [None, None, None]
    if err is None:
        return [None, val, None]
    return [err, val, err]


In [21]:
def _row_get(row, *names):
    """Get a field from a row using stripped-header fallback."""
    stripped_to_actual = {str(c).strip(): c for c in row.index}
    for name in names:
        if name in row.index:
            return row[name]
        key = str(name).strip()
        if key in stripped_to_actual:
            return row[stripped_to_actual[key]]
    return None


def kluska_row_to_record(row):
    """Convert one row of Kluska+2022 Table 1 to the schema dict."""

    # --- names ---
    iras = _clean(_row_get(row, "IRAS"))
    name = _clean(_row_get(row, "Name"))
    if name:
        # strip surrounding [] if present, e.g. "[BD+46.442]" -> "BD+46.442"
        name = re.sub(r"^\[|\]$", "", name).strip() or None

    # Prefer the common name; fall back to IRAS if no common name.
    # The other identifier goes into Notes as an alias.
    if name:
        system_name = name
        alias = iras
    else:
        system_name = iras
        alias = None

    # Drop malformed rows with no identifier at all.
    if system_name is None:
        return None

    # --- coordinates ---
    ra_deg = hms_to_deg(_row_get(row, "RA"))
    dec_deg = dms_to_deg(_row_get(row, "DE", "Dec"))

    # --- period, eccentricity ---
    porb = _to_float(_row_get(row, "Porb", "P_orb", "Period"))
    ecc = _to_float(_row_get(row, "ecc", "Eccentricity"))

    # --- notes: pack the columns the schema doesn't have ---
    transition_disc = _clean(_row_get(row, "yes/no"))
    feh = _to_float(_row_get(row, "[Fe/H]"))
    znti = _to_float(_row_get(row, "[Zn/Ti]"))
    teff = _to_float(_row_get(row, "Teff"))
    lir = _to_float(_row_get(row, "LIR"))
    tturnoff = _to_float(_row_get(row, "TTurnOff"))
    cat = _clean(_row_get(row, "Cat"))
    hks = _clean(_row_get(row, "H-Ks(e)"))
    w1w3 = _clean(_row_get(row, "W1-W3(e)"))
    ref_letter = _clean(_row_get(row, "Ref"))

    note_bits = []
    if alias:                  note_bits.append(f"Alternative names: {alias}")
    # if cat:                    note_bits.append(f"Kluska+2022 disc category: {cat}")
    if transition_disc:        note_bits.append(f"Transition disc (cavity): {transition_disc}")
    if teff is not None:       note_bits.append(f"Teff(post-AGB)={teff:.0f} K")
    # if lir is not None:        note_bits.append(f"L_IR/L_*={lir}%")
    if feh is not None:        note_bits.append(f"[Fe/H]={feh}")
    if znti is not None:       note_bits.append(f"[Zn/Ti]={znti} (depletion proxy)")
    # if tturnoff is not None:   note_bits.append(f"T_turnoff={tturnoff:.0f} K")
    # if hks:                    note_bits.append(f"H-Ks={hks}")
    # if w1w3:                   note_bits.append(f"W1-W3={w1w3}")
    if ref_letter:             note_bits.append(f"Kluska+2022 ref code: {ref_letter}")
    notes = "; ".join(note_bits) if note_bits else None

    # --- SIMBAD lookup URL (coordinate-based) ---
    simbad = None
    if ra_deg is not None and dec_deg is not None:
        simbad = (
            "https://simbad.u-strasbg.fr/simbad/sim-coo"
            f"?Coord={ra_deg:.6f}+{dec_deg:+.6f}&Radius=5&Radius.unit=arcsec"
        )

    return {
        "System Name":      system_name,
        "RA":               _triplet(ra_deg),
        "Dec":              _triplet(dec_deg),
        "Period":           _triplet(porb),
        "Eccentricity":     _triplet(ecc),
        "M1":               _triplet(None),
        "M2":               _triplet(None),
        "Mass Function":    _triplet(None),
        "M1_sin3i":         _triplet(None),
        "M2_sin3i":         _triplet(None),
        "evol_type_1":      "MS",
        "evol_type_2":      "AGB",
        "obs_type_1":       "MS",
        "obs_type_2":       "Post-AGB",
        "system_class":     "Post-AGB binary",
        "Detection Method": ["RV"] if porb is not None else ["SED"],
        "Reference":        ["2022A&A...658A..36K"],
        "Notes":            notes,
        "Simbad":           simbad,
    }

In [22]:
kluska_entries = []
for _, row in kluska_table_full.iterrows():
    rec = kluska_row_to_record(row)
    if rec is not None:
        kluska_entries.append(rec)

print(f"Parsed {len(kluska_entries)} Kluska entries")
display(kluska_entries)


Parsed 85 Kluska entries


[{'System Name': 'BD+46.442',
  'RA': [None, 26.445958333333333, None],
  'Dec': [None, 46.81693611111111, None],
  'Period': [None, 140.82, None],
  'Eccentricity': [None, 0.0, None],
  'M1': [None, None, None],
  'M2': [None, None, None],
  'Mass Function': [None, None, None],
  'M1_sin3i': [None, None, None],
  'M2_sin3i': [None, None, None],
  'evol_type_1': 'MS',
  'evol_type_2': 'AGB',
  'obs_type_1': 'MS',
  'obs_type_2': 'Post-AGB',
  'system_class': 'Post-AGB binary',
  'Detection Method': ['RV'],
  'Reference': ['2022A&A...658A..36K'],
  'Notes': 'Alternative names: IRAS01427+4633; Transition disc (cavity): no; Teff(post-AGB)=6250 K; [Fe/H]=-0.7; [Zn/Ti]=-0.2 (depletion proxy); Kluska+2022 ref code: h',
  'Simbad': 'https://simbad.u-strasbg.fr/simbad/sim-coo?Coord=26.445958++46.816936&Radius=5&Radius.unit=arcsec'},
 {'System Name': 'TW Cam',
  'RA': [None, 65.20041666666667, None],
  'Dec': [None, 57.440555555555555, None],
  'Period': [None, 662.2, None],
  'Eccentricity': [

---
## 2. Oomen et al. 2018

Data hard-coded from the three LaTeX tables in the paper (orbital elements, mass functions, spectroscopic data).  
RA/Dec are queried from SIMBAD.

In [23]:
OOMEN_OUTPUT  = RAW_JSON_DIR / "Oomen2018_postAGB.raw.json"
OOMEN_BIBCODE = "2018A&A...620A..85O"

# ---------- Helper parsers ----------

def pm_to_triplet(value_str):
    """
    Convert 'value±err' strings to [err-, value, err+] triplets.
    Handles upper-only errors like '0.0+0.05' as [0.0, 0.0, 0.05].
    Returns None when the value is missing ('/').
    """
    s = value_str.strip()
    if s in ("/", "") or s.lower() == "nan":
        return None
    s = (s.replace("$", "").replace("\\pm", "±")
          .replace("\\,", "").replace("~", " ").replace("\\", ""))
    # upper-only uncertainty like "0.0+0.05"
    if "+" in s and "±" not in s:
        parts = s.split("+")
        try:
            val  = float(parts[0])
            errp = float(parts[1])
            return [0.0, val, errp]
        except Exception:
            pass
    # standard "value±err"
    if "±" in s:
        val_str, err_str = s.split("±")
        val = float(val_str)
        err = float(err_str)
        return [err, val, err]
    # plain number
    try:
        return [0.0, float(s), 0.0]
    except Exception:
        return None


def as_lower_limit_triplet(value):
    """Encode a lower limit as [0.0, value, +inf]."""
    return [0.0, float(value), float("inf")] if value is not None else None


TRI_NAN = [np.nan, np.nan, np.nan]


def safe_triplet(x):
    """Normalise a triplet-like value; returns NaN triplet if None."""
    if x is None:
        return TRI_NAN.copy()
    try:
        if len(x) == 3:
            return [float(x[0]), float(x[1]), float(x[2])]
    except Exception:
        pass
    try:
        return [np.nan, float(x), np.nan]
    except Exception:
        return TRI_NAN.copy()


columns = [
    "System Name", "RA", "Dec", "Period", "Eccentricity",
    "M1", "M1_sin3i", "M2", "M2_sin3i", "q", "Mass Function",
    "Type1", "Type2", "Detection Method", "Reference", "Notes"
]
oomen_df = pd.DataFrame(columns=columns)


def add_observation(df, system_name, ra, dec, period, ecc,
                    m1, m1_sin3i, m2, m2_sin3i, q, mass_func,
                    type1, type2, method, reference, notes=""):
    new_row = {
        "System Name": system_name, "RA": ra, "Dec": dec,
        "Period": period, "Eccentricity": ecc,
        "M1": m1, "M1_sin3i": m1_sin3i, "M2": m2, "M2_sin3i": m2_sin3i,
        "q": q, "Mass Function": mass_func,
        "Type1": type1, "Type2": type2,
        "Detection Method": method, "Reference": reference, "Notes": notes,
    }
    return pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

In [24]:
# ---------- Table 1: orbital elements ----------
tab1_rows = [
    (1,  "89 Her",          "289.1±0.2",    "0.29±0.07",   "1"),
    (2,  "AC Her",          "1188.9±1.2",   "0.0+0.05",    "2"),
    (3,  "BD+39 4926",      "871.7±0.4",    "0.024±0.006", "3"),
    (4,  "BD+46 442",       "140.82±0.02",  "0.085±0.005", "4"),
    (5,  "DY Ori",          "1248±36",      "0.22±0.08",   "5"),
    (6,  "EP Lyr",          "1151±14",      "0.39±0.09",   "5"),
    (7,  "HD 44179",        "317.6±1.1",    "0.27±0.03",   "6,7"),
    (8,  "HD 46703",        "597.4±0.2",    "0.30±0.02",   "8"),
    (9,  "HD 52961",        "1288.6±0.3",   "0.23±0.01",   "7"),
    (10, "HD 95767",        "1989±61",      "0.25±0.05",   "9"),
    (11, "HD 108015",       "906.3±5.9",    "0.0+0.03",    "9"),
    (12, "HD 131356",       "1488.0±8.7",   "0.32±0.04",   "9"),
    (13, "HD 158616",       "363.3±1.0",    "0.0+0.1",     "10"),
    (14, "HD 213985",       "259.6±0.7",    "0.21±0.05",   "9"),
    (15, "HP Lyr",          "1818±80",      "0.20±0.04",   "5"),
    (16, "HR 4049",         "430.6±0.1",    "0.30±0.01",   "11"),
    (17, "IRAS 05208-2035", "234.38±0.04",  "0.0+0.02",    "17"),
    (18, "IRAS 06165+3158", "262.6±0.7",    "0.0+0.05",    ""),
    (19, "IRAS 06452-3456", "215.4±0.4",    "0.0+0.03",    ""),
    (20, "IRAS 08544-4431", "501.1±1.0",    "0.20±0.02",   "12"),
    (21, "IRAS 09144-4933", "1762±27",      "0.30±0.04",   "5"),
    (22, "IRAS 15469-5311", "390.2±0.7",    "0.08±0.02",   "12"),
    (23, "IRAS 16230-3410", "649.8±3.5",    "0.0+0.13",    ""),
    (24, "IRAS 17038-4815", "1394±12",      "0.63±0.06",   "5"),
    (25, "IRAS 19125+0343", "519.7±0.7",    "0.24±0.03",   "12"),
    (26, "IRAS 19135+3937", "126.97±0.08",  "0.13±0.03",   "13"),
    (27, "IRAS 19157-0247", "119.6±0.1",    "0.34±0.04",   "12"),
    (28, "RU Cen",          "1489±10",      "0.62±0.07",   "14"),
    (29, "SAO 173329",      "115.951±0.002","0.0+0.04",    "9"),
    (30, "ST Pup",          "406.0±2.2",    "0.0+0.04",    "15"),
    (31, "SX Cen",          "564.3±7.6",    "0.0+0.06",    "14"),
    (32, "TW Cam",          "662.2±5.3",    "0.25±0.04",   "5"),
    (33, "U Mon",           "2550±143",     "0.25±0.06",   "16"),
]
tab1 = {name: {"Period": pm_to_triplet(per), "Eccentricity": pm_to_triplet(ecc)}
        for _, name, per, ecc, _ in tab1_rows}

# ---------- Table 2: mass functions and minimum companion masses ----------
tab2_rows = [
    (1,  "89 Her",          "0.106±0.007",  "0.0019±0.0004",  "0.10"),
    (2,  "AC Her",          "1.176±0.080",  "0.153±0.032",    "0.64"),
    (3,  "BD+39 4926",      "1.286±0.007",  "0.373±0.006",    "1.03"),
    (4,  "BD+46 442",       "0.3074±0.0014","0.195±0.003",    "0.72"),
    (5,  "DY Ori",          "1.39±0.11",    "0.23±0.05",      "0.79"),
    (6,  "EP Lyr",          "1.30±0.12",    "0.22±0.06",      "0.77"),
    (7,  "HD 44179",        "0.342±0.008",  "0.053±0.003",    "0.38"),
    (8,  "HD 46703",        "0.839±0.015",  "0.220±0.012",    "0.77"),
    (9,  "HD 52961",        "1.507±0.034",  "0.274±0.019",    "0.87"),
    (10, "HD 95767",        "2.14±0.16",    "0.33±0.07",      "0.96"),
    (11, "HD 108015",       "0.28±0.02",    "0.0036±0.0009",  "0.13"),
    (12, "HD 131356",       "2.11±0.09",    "0.57±0.07",      "1.33"),
    (13, "HD 158616",       "0.28±0.03",    "0.022±0.008",    "0.26"),
    (14, "HD 213985",       "0.733±0.025",  "0.777±0.079",    "1.62"),
    (15, "HP Lyr",          "1.27±0.06",    "0.083±0.007",    "0.47"),
    (16, "HR 4049",         "0.627±0.010",  "0.177±0.008",    "0.69"),
    (17, "IRAS 05208-2035", "0.396±0.004",  "0.150±0.005",    "0.64"),
    (18, "IRAS 06165+3158", "0.374±0.011",  "0.10±0.01",      "0.52"),
    (19, "IRAS 06452-3456", "0.73±0.01",    "1.12±0.05",      "2.07"),
    (20, "IRAS 08544-4431", "0.398±0.008",  "0.033±0.002",    "0.31"),
    (21, "IRAS 09144-4933", "2.25±0.11",    "0.49±0.07",      "1.21"),
    (22, "IRAS 15469-5311", "0.438±0.015",  "0.074±0.008",    "0.45"),
    (23, "IRAS 16230-3410", "0.232±0.021",  "0.004±0.001",    "0.13"),
    (24, "IRAS 17038-4815", "1.52±0.08",    "0.24±0.04",      "0.81"),
    (25, "IRAS 19125+0343", "0.56±0.02",    "0.086±0.010",    "0.48"),
    (26, "IRAS 19135+3937", "0.209±0.008",  "0.075±0.008",    "0.45"),
    (27, "IRAS 19157-0247", "0.083±0.003",  "0.0053±0.0007",  "0.15"),
    (28, "RU Cen",          "2.38±0.15",    "0.81±0.17",      "1.66"),
    (29, "SAO 173329",      "0.132±0.003",  "0.023±0.001",    "0.27"),
    (30, "ST Pup",          "0.67±0.02",    "0.241±0.026",    "0.81"),
    (31, "SX Cen",          "1.12±0.05",    "0.58±0.07",      "1.35"),
    (32, "TW Cam",          "0.83±0.04",    "0.174±0.022",    "0.68"),
    (33, "U Mon",           "3.38±0.31",    "0.79±0.18",      "1.64"),
]
tab2 = {name: {"Mass Function": pm_to_triplet(f), "M2_min": float(m2min)}
        for _, name, _, f, m2min in tab2_rows}

# ---------- Table 3: spectroscopic / depletion data (used for Notes only) ----------
tab3_rows = [
    (1,  "89 Her",          "6600", "0.8",  "0.02", "0.38",  "-0.5", "mild"),
    (2,  "AC Her",          "5800", "1.0",  "0.46", "0.24",  "-1.4", "mild"),
    (3,  "BD+39 4926",      "7750", "1.0",  "0.23", "0.0",   "-2.4", "strong"),
    (4,  "BD+46 442",       "6250", "1.5",  "0.23", "0.19",  "-0.8", "no"),
    (5,  "DY Ori",          "5900", "1.5",  "0.90", "0.74",  "-2.3", "strong"),
    (6,  "EP Lyr",          "6200", "1.5",  "0.48", "0.04",  "-1.8", "moderate"),
    (7,  "HD 44179",        "7500", "0.8",  "0.15", "18.1",  "-3.3", "strong"),
    (8,  "HD 46703",        "6250", "1.0",  "0.23", "0.02",  "-1.7", "mild"),
    (9,  "HD 52961",        "6000", "0.5",  "0.04", "0.13",  "-4.8", "strong"),
    (10, "HD 95767",        "7500", "2.0",  "0.58", "0.55",  "0.1",  "no"),
    (11, "HD 108015",       "7000", "1.5",  "0.15", "1.04",  "-0.1", "no"),
    (12, "HD 131356",       "6000", "1.0",  "0.15", "0.65",  "0.0",  "mild"),
    (13, "HD 158616",       "7250", "1.25", "0.51", "0.23",  "-0.6", "no"),
    (14, "HD 213985",       "8250", "1.5",  "0.12", "0.35",  "-0.9", "strong"),
    (15, "HP Lyr",          "6300", "1.0",  "0.39", "0.56",  "-1.0", "strong"),
    (16, "HR 4049",         "7600", "1.1",  "0.20", "0.12",  "-4.8", "strong"),
    (17, "IRAS 05208-2035", "4250", "0.75", "0.01", "0.43",  "-0.7", "no"),
    (18, "IRAS 06165+3158", "4250", "1.5",  "0.53", "0.39",  "-0.9", "no"),
    (19, "IRAS 06452-3456", "/",   "/",    "0.94", "0.11",  "/",    "/"),
    (20, "IRAS 08544-4431", "7250", "1.5",  "1.32", "0.49",  "-0.3", "mild"),
    (21, "IRAS 09144-4933", "5750", "0.5",  "1.78", "0.81",  "-0.3", "moderate"),
    (22, "IRAS 15469-5311", "7500", "1.5",  "1.27", "0.74",  "0.0",  "strong"),
    (23, "IRAS 16230-3410", "6250", "1.0",  "0.72", "0.46",  "-0.7", "moderate"),
    (24, "IRAS 17038-4815", "4750", "0.5",  "0.57", "0.79",  "-1.5", "no"),
    (25, "IRAS 19125+0343", "7750", "1.0",  "0.94", "0.90",  "-0.3", "strong"),
    (26, "IRAS 19135+3937", "6000", "0.5",  "0.28", "0.26",  "-1.0", "no"),
    (27, "IRAS 19157-0247", "7750", "1.0",  "0.66", "0.79",  "0.1",  "no"),
    (28, "RU Cen",          "6000", "1.5",  "0.18", "0.39",  "-1.9", "moderate"),
    (29, "SAO 173329",      "7000", "1.0",  "0.31", "0.35",  "-0.9", "no"),
    (30, "ST Pup",          "5500", "1.0",  "0.06", "1.32",  "-1.5", "strong"),
    (31, "SX Cen",          "6250", "1.5",  "0.17", "0.40",  "-1.1", "strong"),
    (32, "TW Cam",          "4800", "0.0",  "0.42", "0.43",  "-0.5", "no"),
    (33, "U Mon",           "5000", "0.0",  "0.34", "0.28",  "-0.8", "no"),
]
tab3 = {
    name: {"EBV": None if ebv == "/" else float(ebv),
           "LIR": None if lir == "/" else float(lir),
           "Depletion": depl}
    for _, name, _teff, _logg, ebv, lir, _feh, depl in tab3_rows
}

# ---------- Build the DataFrame ----------
for name in [r[1] for r in tab1_rows]:
    per  = safe_triplet(tab1[name]["Period"])
    ecc  = safe_triplet(tab1[name]["Eccentricity"])
    mf   = safe_triplet(tab2[name]["Mass Function"])
    m1ll = as_lower_limit_triplet(tab2[name]["M2_min"]) # their m2 is our m1

    notes_bits = []
    if name in tab3:
        ebv  = tab3[name]["EBV"]
        lir  = tab3[name]["LIR"]
        depl = tab3[name]["Depletion"]
        if ebv  is not None:         notes_bits.append(f"E(B-V)={ebv}")
        if lir  is not None:         notes_bits.append(f"L_IR/L_*={lir}")
        if depl and depl != "/":    notes_bits.append(f"Depletion={depl}")
    notes_bits.append("Minimum accretor mass assumes donor M2=0.6 Msun and i=75 deg (from Table 2).")
    notes = "; ".join(notes_bits)

    oomen_df = add_observation(
        oomen_df,
        system_name=name,
        ra=TRI_NAN.copy(),
        dec=TRI_NAN.copy(),
        period=per, ecc=ecc,
        m1=m1ll, m1_sin3i=TRI_NAN.copy(),
        m2=TRI_NAN.copy(), m2_sin3i=TRI_NAN.copy(),
        q=TRI_NAN.copy(), mass_func=mf,
        type1="MS?", type2="post AGB",
        method=["RV"],
        reference=[OOMEN_BIBCODE],
        notes=notes,
    )

print(f"Built Oomen DataFrame: {len(oomen_df)} rows")
display(oomen_df[["System Name", "Period", "Eccentricity", "Mass Function", "M2"]])

Built Oomen DataFrame: 33 rows


,System Name,Period,Eccentricity,Mass Function,M2
0,89 Her,"[0.2, 289.1, 0.2]","[0.07, 0.29, 0.07]","[0.0004, 0.0019, 0.0004]","[nan, nan, nan]"
1,AC Her,"[1.2, 1188.9, 1.2]","[0.0, 0.0, 0.05]","[0.032, 0.153, 0.032]","[nan, nan, nan]"
2,BD+39 4926,"[0.4, 871.7, 0.4]","[0.006, 0.024, 0.006]","[0.006, 0.373, 0.006]","[nan, nan, nan]"
3,BD+46 442,"[0.02, 140.82, 0.02]","[0.005, 0.085, 0.005]","[0.003, 0.195, 0.003]","[nan, nan, nan]"
4,DY Ori,"[36.0, 1248.0, 36.0]","[0.08, 0.22, 0.08]","[0.05, 0.23, 0.05]","[nan, nan, nan]"
5,EP Lyr,"[14.0, 1151.0, 14.0]","[0.09, 0.39, 0.09]","[0.06, 0.22, 0.06]","[nan, nan, nan]"
6,HD 44179,"[1.1, 317.6, 1.1]","[0.03, 0.27, 0.03]","[0.003, 0.053, 0.003]","[nan, nan, nan]"
7,HD 46703,"[0.2, 597.4, 0.2]","[0.02, 0.3, 0.02]","[0.012, 0.22, 0.012]","[nan, nan, nan]"
8,HD 52961,"[0.3, 1288.6, 0.3]","[0.01, 0.23, 0.01]","[0.019, 0.274, 0.019]","[nan, nan, nan]"
9,HD 95767,"[61.0, 1989.0, 61.0]","[0.05, 0.25, 0.05]","[0.07, 0.33, 0.07]","[nan, nan, nan]"


In [25]:
# ---------- Query SIMBAD for RA/Dec ----------
targets = oomen_df["System Name"].tolist()


def normalize_lookup_name(name):
    """Normalize object names for stable SIMBAD result matching."""
    if name is None:
        return ""
    s = str(name).upper().strip()
    return re.sub(r'[\s._~;]', '', s)

result = subprocess.run(
    ["python3", "Get_Coords_From_SIMBAD.py"] + targets,
    capture_output=True, text=True
)

if result.returncode != 0:
    raise RuntimeError(result.stderr.strip() or "SIMBAD coordinate lookup failed")

RADEC_data = {}
for dict_str in re.findall(r"=\s*({.*?})", result.stdout, re.DOTALL):
    entry = ast.literal_eval(dict_str)
    key = normalize_lookup_name(entry.get("System Name"))
    if key:
        RADEC_data[key] = entry


def lookup_coord_triplet(target, field):
    entry = RADEC_data.get(normalize_lookup_name(target))
    if entry is None:
        return TRI_NAN.copy()
    return entry.get(field, TRI_NAN.copy())


RA_values  = [lookup_coord_triplet(target, "RA") for target in targets]
DEC_values = [lookup_coord_triplet(target, "Dec") for target in targets]

oomen_df["RA"]  = RA_values
oomen_df["Dec"] = DEC_values

# Use the single Oomen bibcode for all rows
oomen_df["Reference"] = [[OOMEN_BIBCODE]] * len(oomen_df)

missing_targets = [target for target in targets if normalize_lookup_name(target) not in RADEC_data]
print(f"RA/Dec populated for {len(targets) - len(missing_targets)} of {len(targets)} systems")
if missing_targets:
    print("Missing SIMBAD matches:", missing_targets)
display(oomen_df)

RA/Dec populated for 9 of 33 systems
Missing SIMBAD matches: ['89 Her', 'AC Her', 'DY Ori', 'EP Lyr', 'HD 46703', 'HD 52961', 'HD 95767', 'HD 131356', 'HD 158616', 'HP Lyr', 'HR 4049', 'IRAS 05208-2035', 'IRAS 08544-4431', 'IRAS 16230-3410', 'IRAS 17038-4815', 'IRAS 19125+0343', 'IRAS 19135+3937', 'IRAS 19157-0247', 'RU Cen', 'SAO 173329', 'ST Pup', 'SX Cen', 'TW Cam', 'U Mon']


,System Name,RA,Dec,Period,Eccentricity,M1,M1_sin3i,M2,M2_sin3i,q,Mass Function,Type1,Type2,Detection Method,Reference,Notes
0,89 Her,"[nan, nan, nan]","[nan, nan, nan]","[0.2, 289.1, 0.2]","[0.07, 0.29, 0.07]","[0.0, 0.1, inf]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[0.0004, 0.0019, 0.0004]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.02; L_IR/L_*=0.38; Depletion=mild; Mi...
1,AC Her,"[nan, nan, nan]","[nan, nan, nan]","[1.2, 1188.9, 1.2]","[0.0, 0.0, 0.05]","[0.0, 0.64, inf]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[0.032, 0.153, 0.032]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.46; L_IR/L_*=0.24; Depletion=mild; Mi...
2,BD+39 4926,"[8.03e-06, 341.54678, 8.03e-06]","[6.53e-06, 40.107308, 6.53e-06]","[0.4, 871.7, 0.4]","[0.006, 0.024, 0.006]","[0.0, 1.03, inf]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[0.006, 0.373, 0.006]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.23; L_IR/L_*=0.0; Depletion=strong; M...
3,BD+46 442,"[6.86e-06, 26.445963, 6.86e-06]","[3.42e-06, 46.816927, 3.42e-06]","[0.02, 140.82, 0.02]","[0.005, 0.085, 0.005]","[0.0, 0.72, inf]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[0.003, 0.195, 0.003]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.23; L_IR/L_*=0.19; Depletion=no; Mini...
4,DY Ori,"[nan, nan, nan]","[nan, nan, nan]","[36.0, 1248.0, 36.0]","[0.08, 0.22, 0.08]","[0.0, 0.79, inf]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[0.05, 0.23, 0.05]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.9; L_IR/L_*=0.74; Depletion=strong; M...
5,EP Lyr,"[nan, nan, nan]","[nan, nan, nan]","[14.0, 1151.0, 14.0]","[0.09, 0.39, 0.09]","[0.0, 0.77, inf]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[0.06, 0.22, 0.06]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.48; L_IR/L_*=0.04; Depletion=moderate...
6,HD 44179,"[0.00050026, 94.992577, 0.00050026]","[0.00040556, -10.637418, 0.00040556]","[1.1, 317.6, 1.1]","[0.03, 0.27, 0.03]","[0.0, 0.38, inf]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[0.003, 0.053, 0.003]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.15; L_IR/L_*=18.1; Depletion=strong; ...
7,HD 46703,"[nan, nan, nan]","[nan, nan, nan]","[0.2, 597.4, 0.2]","[0.02, 0.3, 0.02]","[0.0, 0.77, inf]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[0.012, 0.22, 0.012]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.23; L_IR/L_*=0.02; Depletion=mild; Mi...
8,HD 52961,"[nan, nan, nan]","[nan, nan, nan]","[0.3, 1288.6, 0.3]","[0.01, 0.23, 0.01]","[0.0, 0.87, inf]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[0.019, 0.274, 0.019]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.04; L_IR/L_*=0.13; Depletion=strong; ...
9,HD 95767,"[nan, nan, nan]","[nan, nan, nan]","[61.0, 1989.0, 61.0]","[0.05, 0.25, 0.05]","[0.0, 0.96, inf]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[nan, nan, nan]","[0.07, 0.33, 0.07]",MS?,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.58; L_IR/L_*=0.55; Depletion=no; Mini...


---
## 3. Moltzer et al. 2025

https://ui.adsabs.harvard.edu/abs/2025A%26A...703A.294M/abstract

Compiled catalog of Galactic post-AGB and post-RGB binaries, parsed from `Moltzer2025.tex` (Table 1).  
Includes both spectroscopically and photometrically determined orbital periods.  
RA/Dec are not provided in the source table — set to `[None, None, None]`.

In [26]:
MOLTZER_INPUT   = DATA_DIR / "latex_input_data" / "Moltzer2025.tex"
MOLTZER_OUTPUT  = RAW_JSON_DIR / "Moltzer2025_postAGB.raw.json"

# Reference number -> ADS bibcode (update placeholder strings as needed)
MOLTZER_REF_MAP = {
    "1":  "Csornyei2019",          # AU Peg
    "2":  "2018A&A...620A..85O",   # Oomen+2018
    "3":  "Bolton1980",            # HD 137569
    "4":  "Maas2003",              # HD 93662
    "5":  "Manick2019",            # RV Tau, HP Lyr, ...
    "6":  "Manick2021",            # V510 Pup
    "7":  "Kiss2007",              # photometric
    "8":  "2022A&A...658A..36K",   # Kluska+2022
    "9":  "Bodi2019",              # BT Lac
    "10": "Percy2015",             # BT Lac, R Sge
    "11": "Kiss2017",              # photometric
    "12": "Horne2012",             # RS Sge
    "13": "Hrivnak2024",           # QY Sge
}


def parse_moltzer_refs(ref_str):
    """Convert '(2)' or '(9)(10)' to a list of bibcodes."""
    nums = re.findall(r'\((\d+)\)', ref_str)
    return [MOLTZER_REF_MAP.get(n, f"ref_{n}") for n in nums]


def parse_pm_or_none(raw_str):
    """pm_to_triplet wrapper: returns [None, val, None] when no error is stated."""
    s = raw_str.strip()
    if not s or s == "...":
        return None
    has_err = ("\\pm" in s or "±" in s)
    tri = pm_to_triplet(s)
    if tri is None:
        return None
    return tri if has_err else [None, tri[1], None]


def normalize_moltzer_iras_id(raw_name):
    """Canonicalize IRAS-like identifiers so they merge across catalogs."""
    s = raw_name.strip()
    if not s or s == "...":
        return None

    # Moltzer sometimes lists IRAS systems as bare numeric ids like 09144-4933.
    s = re.sub(r'^F(?=IRAS|\d)', '', s, flags=re.IGNORECASE)
    match = re.fullmatch(r'(?:IRAS\s*)?(\d{5}[+-]\d{4})', s, flags=re.IGNORECASE)
    if match:
        return f"IRAS {match.group(1)}"
    return s


def parse_moltzer_tex(filepath):
    entries = []
    with open(filepath, "r", encoding="utf-8") as fh:
        lines = fh.readlines()

    for line in lines:
        line = line.strip()
        # Skip non-data lines
        if not line or line.count("&") != 4:
            continue
        if any(line.startswith(kw) for kw in (
            "\\hline", "\\multicolumn", "\\tablef", "\\tablebib",
            "\\end", "\\begin", "\\caption", "\\label", "\\centering",
        )):
            continue

        parts      = [p.strip() for p in line.split("&")]
        iras_raw   = parts[0]
        name_raw   = parts[1]
        period_raw = parts[2]
        ecc_raw    = parts[3]
        ref_raw    = parts[4].replace("\\\\", "").strip()

        # Skip header lines that otherwise match the generic data pattern.
        if iras_raw.lower() == "iras" and name_raw.lower() == "name":
            continue

        # IRAS id (strip leading F, ignore "...")
        iras_id = normalize_moltzer_iras_id(iras_raw)

        # Alternative names: split on ';', drop "..."
        alt_names = [n.strip() for n in name_raw.split(";")
                     if n.strip() and n.strip() != "..."]

        # Period and eccentricity
        period = parse_pm_or_none(period_raw)
        is_photometric = ecc_raw.strip() in ("...", "")
        ecc    = None if is_photometric else parse_pm_or_none(ecc_raw)

        refs = parse_moltzer_refs(ref_raw)

        all_names = ([iras_id] if iras_id else []) + alt_names
        if not all_names:
            continue
        system_name = all_names if len(all_names) > 1 else all_names[0]

        entries.append({
            "System Name":      system_name,
            "RA":               [None, None, None],
            "Dec":              [None, None, None],
            "Period":           period if period else [None, None, None],
            "Eccentricity":     ecc    if ecc    else [None, None, None],
            "M1":               [None, None, None],
            "M2":               [None, None, None],
            "Mass Function":    [None, None, None],
            "M1_sin3i":         [None, None, None],
            "M2_sin3i":         [None, None, None],
            "evol_type_1":      "MS",
            "evol_type_2":      "AGB",
            "obs_type_1":       "MS",
            "obs_type_2":       "Post-AGB",
            "system_class":     "Post-AGB binary",
            "Detection Method": ["Photometric"] if is_photometric else ["RV"],
            "Reference":        refs,
            "Notes":            "Source: Moltzer+2025 Table 1.",
            "Simbad":           None,
        })
    return entries


moltzer_entries = parse_moltzer_tex(MOLTZER_INPUT)
print(f"Parsed {len(moltzer_entries)} Moltzer+2025 entries")
print(json.dumps(moltzer_entries[0], indent=2) if moltzer_entries else "No entries parsed")

Parsed 53 Moltzer+2025 entries
{
  "System Name": [
    "IRAS 21216+1803",
    "AU Peg"
  ],
  "RA": [
    null,
    null,
    null
  ],
  "Dec": [
    null,
    null,
    null
  ],
  "Period": [
    0.0003,
    53.3344,
    0.0003
  ],
  "Eccentricity": [
    0.003,
    0.043,
    0.003
  ],
  "M1": [
    null,
    null,
    null
  ],
  "M2": [
    null,
    null,
    null
  ],
  "Mass Function": [
    null,
    null,
    null
  ],
  "M1_sin3i": [
    null,
    null,
    null
  ],
  "M2_sin3i": [
    null,
    null,
    null
  ],
  "evol_type_1": "MS",
  "evol_type_2": "AGB",
  "obs_type_1": "MS",
  "obs_type_2": "Post-AGB",
  "system_class": "Post-AGB binary",
  "Detection Method": [
    "RV"
  ],
  "Reference": [
    "Csornyei2019"
  ],
  "Notes": "Source: Moltzer+2025 Table 1.",
  "Simbad": null
}


---
## 4. Export all three catalogs to raw JSON

In [27]:
def _triplet_to_json_value(tri):
    """Convert a triplet to [err-, value, err+] with JSON-safe nulls."""
    if tri is None:
        return [None, None, None]
    if not isinstance(tri, (list, tuple, np.ndarray)):
        try:
            v = float(tri)
            return [None, v if np.isfinite(v) else None, None]
        except Exception:
            return [None, None, None]
    out = []
    for x in list(tri)[:3]:
        try:
            xv = float(x)
            out.append(xv if np.isfinite(xv) else None)
        except Exception:
            out.append(None)
    while len(out) < 3:
        out.append(None)
    return out


def _listify(v):
    if v is None:
        return []
    if isinstance(v, list):
        return v
    if isinstance(v, tuple):
        return list(v)
    return [v]


def convert_oomen_df_to_entries(df):
    entries = []
    for _, row in df.iterrows():
        notes   = row.get("Notes", None)
        raw_m1  = row.get("M1", None)
        m1_tri  = _triplet_to_json_value(raw_m1)

        # +inf upper error means this is a lower limit - capture that in Notes
        try:
            if (isinstance(raw_m1, (list, tuple, np.ndarray)) and len(raw_m1) == 3
                    and not np.isfinite(float(raw_m1[2]))):
                extra = "M1 stored as lower limit from Oomen+2018 Table 2."
                notes = f"{notes}; {extra}" if notes else extra
        except Exception:
            pass

        entries.append({
            "System Name":    row.get("System Name", None),
            "RA":             _triplet_to_json_value(row.get("RA",           None)),
            "Dec":            _triplet_to_json_value(row.get("Dec",          None)),
            "Period":         _triplet_to_json_value(row.get("Period",       None)),
            "Eccentricity":   _triplet_to_json_value(row.get("Eccentricity", None)),
            "M1":             m1_tri,
            "M2":             _triplet_to_json_value(row.get("M2",           None)),
            "Mass Function":  _triplet_to_json_value(row.get("Mass Function",None)),
            "M1_sin3i":       _triplet_to_json_value(row.get("M1_sin3i",     None)),
            "M2_sin3i":       _triplet_to_json_value(row.get("M2_sin3i",     None)),
            "evol_type_1":    "MS",
            "evol_type_2":    "AGB",
            "obs_type_1":     "MS",
            "obs_type_2":     "Post-AGB",
            "system_class":   "Post-AGB binary",
            "Detection Method": _listify(row.get("Detection Method", [])),
            "Reference":      _listify(row.get("Reference",          [])),
            "Notes":          notes,
            "Simbad":         None,
        })
    return entries


def save_raw_json(entries, output_file):
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as fh:
        json.dump(entries, fh, separators=(",", ": "), ensure_ascii=False, indent=None)
    print(f"Saved {len(entries)} entries to {output_file}")


# # --- Save Kluska ---
# save_raw_json(kluska_entries, KLUSKA_OUTPUT)
# print(json.dumps(kluska_entries[0], indent=2) if kluska_entries else "")

# # --- Save Oomen ---
oomen_entries = convert_oomen_df_to_entries(oomen_df)
# save_raw_json(oomen_entries, OOMEN_OUTPUT)
# print(json.dumps(oomen_entries[0], indent=2) if oomen_entries else "")

# # --- Save Moltzer ---
# save_raw_json(moltzer_entries, MOLTZER_OUTPUT)
# print(json.dumps(moltzer_entries[0], indent=2) if moltzer_entries else "")

---
## 5. Cross-catalog overlap

Find systems that appear in two or more of the three catalogs by comparing normalized name variants.  
Uses union-find to group entries that share any name (IRAS id, common name, or alternative designation).

In [28]:
from collections import defaultdict

# --- Build catalogs dict and prepare for overlap detection ---
catalogs = {
    "Kluska": kluska_entries,
    "Oomen": oomen_entries,
    "Moltzer": moltzer_entries,
}

print(f"Catalog sizes: Kluska={len(kluska_entries)}, Oomen={len(oomen_entries)}, Moltzer={len(moltzer_entries)}")


def normalize_name(name):
    """Standardize a name to uppercase, remove IRAS prefix, strip punctuation."""
    if not name:
        return ""
    s = str(name).upper().strip()
    # Remove leading F from F-type IRAS
    s = re.sub(r'^F(?=IRAS)', '', s)
    # Remove punctuation and extra spaces
    s = re.sub(r'[\s._~;]', '', s)
    return s


def get_all_names(entry):
    """Extract all name variants from System Name (handles lists and strings)."""
    sn = entry.get("System Name")
    if not sn:
        return []
    if isinstance(sn, list):
        return [n for n in sn if n]
    if isinstance(sn, str):
        # Split on semicolon AND comma
        return [n.strip() for n in re.split(r'[;,]', sn) if n.strip()]
    return [str(sn)]


def extract_coords(entry):
    """Pull (RA, Dec) as floats from triplet format, handling None values."""
    ra = entry.get("RA", [None, None, None])
    dec = entry.get("Dec", [None, None, None])
    try:
        ra_val = ra[1] if isinstance(ra, list) and len(ra) > 1 else ra
        dec_val = dec[1] if isinstance(dec, list) and len(dec) > 1 else dec
        if ra_val is not None:
            ra_val = float(ra_val)
        if dec_val is not None:
            dec_val = float(dec_val)
        return (ra_val, dec_val) if ra_val is not None and dec_val is not None else None
    except (TypeError, ValueError):
        return None


def coords_match(c1, c2, tol_deg=1/3600):
    """Check if two (RA, Dec) coordinates match within tolerance (default 1 arcsec)."""
    if c1 is None or c2 is None:
        return False
    try:
        ra1, dec1 = float(c1[0]), float(c1[1])
        ra2, dec2 = float(c2[0]), float(c2[1])
        if not (np.isfinite(ra1) and np.isfinite(dec1) and np.isfinite(ra2) and np.isfinite(dec2)):
            return False
        dist = np.sqrt((ra1 - ra2)**2 + (dec1 - dec2)**2)
        return dist < tol_deg
    except (TypeError, ValueError):
        return False


# --- Union-find for grouping duplicates by name and coordinate matching ---
parent = {}
rank = {}

def uf_find(x):
    if x not in parent:
        parent[x] = x
        rank[x] = 0
    if parent[x] != x:
        parent[x] = uf_find(parent[x])
    return parent[x]

def uf_union(a, b):
    ra, rb = uf_find(a), uf_find(b)
    if ra == rb:
        return
    if rank[ra] < rank[rb]:
        parent[ra] = rb
    elif rank[ra] > rank[rb]:
        parent[rb] = ra
    else:
        parent[rb] = ra
        rank[ra] += 1

# --- Build all_keys and name_to_keys for single-pass union-find ---
all_keys = []
name_to_keys = defaultdict(list)

for src, entries_list in catalogs.items():
    for idx, entry in enumerate(entries_list):
        key = (src, idx)
        all_keys.append(key)
        for name in get_all_names(entry):
            norm = normalize_name(name)
            if norm:
                name_to_keys[norm].append(key)

# --- First pass: union by name match ---
for norm_name, keys_with_name in name_to_keys.items():
    for i in range(1, len(keys_with_name)):
        uf_union(keys_with_name[0], keys_with_name[i])

# --- Second pass: union by coordinate match (within same union-find group) ---
for i, key_i in enumerate(all_keys):
    coords_i = extract_coords(catalogs[key_i[0]][key_i[1]])
    if coords_i is None:
        continue
    for key_j in all_keys[i+1:]:
        coords_j = extract_coords(catalogs[key_j[0]][key_j[1]])
        if coords_j is None:
            continue
        if coords_match(coords_i, coords_j):
            uf_union(key_i, key_j)

print(f"Initialized union-find with {len(all_keys)} entries")


Catalog sizes: Kluska=85, Oomen=33, Moltzer=53
Initialized union-find with 171 entries


In [29]:

# Group entries by union-find root
groups_map = defaultdict(list)
for key in all_keys:
    root = uf_find(key)
    groups_map[root].append(key)

print(f"DEBUG: groups_map has {len(groups_map)} groups total")
print(f"DEBUG: Groups with >1 entry: {sum(1 for g in groups_map.values() if len(g) > 1)}")

# Keep only groups that span ≥2 catalogs
multi_groups = sorted(
    [g for g in groups_map.values() if len({src for src, _ in g}) >= 2],
    key=lambda g: min(catalogs[src][idx].get("Period", [None, None, None])[1] or 0
                      for src, idx in g)
)

DEBUG: groups_map has 87 groups total
DEBUG: Groups with >1 entry: 53


In [30]:
def has_value(triplet):
    """True when a [err-, value, err+] triplet has a finite center value."""
    if not isinstance(triplet, (list, tuple)) or len(triplet) < 2:
        return False
    try:
        return triplet[1] is not None and np.isfinite(float(triplet[1]))
    except Exception:
        return False


def _to_name_list(system_name):
    if isinstance(system_name, list):
        return [str(x).strip() for x in system_name if str(x).strip()]
    if system_name is None:
        return []
    s = str(system_name).strip()
    return [s] if s else []


def _pick_primary_name(names):
    """Choose one canonical name, preferring IRAS-style identifiers."""
    for n in names:
        if re.match(r'^(?:F)?IRAS\s*\d{5}[+-]\d{4}$', n, flags=re.IGNORECASE):
            return re.sub(r'^F(?=IRAS)', '', n, flags=re.IGNORECASE).strip()
    return names[0]


def _finalize_system_name_and_notes(entry):
    """Ensure System Name is a single string and aliases move to Notes."""
    names = []
    for n in _to_name_list(entry.get("System Name")):
        if n not in names:
            names.append(n)
    if not names:
        return entry

    primary = _pick_primary_name(names)
    alt_names = [n for n in names if n != primary]

    entry["System Name"] = primary
    if alt_names:
        alt_note = f"alternative names: {', '.join(alt_names)}"
        note = entry.get("Notes")
        if note:
            if alt_note not in note:
                entry["Notes"] = f"{note}; {alt_note}"
        else:
            entry["Notes"] = alt_note
    return entry


def _pick_first_value(entries, field, fallback=[None, None, None]):
    for e in entries:
        v = e.get(field)
        if has_value(v):
            return v
    return fallback


def _pick_by_source_priority(group, field, source_priority, catalogs, fallback=[None, None, None]):
    """Pick a triplet field by explicit source priority within one merged group."""
    by_source = {}
    for src, idx in group:
        by_source[src] = catalogs[src][idx]
    for src in source_priority:
        e = by_source.get(src)
        if e is None:
            continue
        v = e.get(field)
        if has_value(v):
            return v
    return fallback


priority = {"Oomen": 0, "Kluska": 1, "Moltzer": 2}
merged_entries = []
included_keys = set()

# Merge only groups that appear in >=2 catalogs.
for group in multi_groups:
    ordered = sorted(group, key=lambda k: priority.get(k[0], 99))
    group_entries = [catalogs[src][idx] for src, idx in ordered]

    names = []
    refs = []
    methods = []
    notes_bits = []

    for e in group_entries:
        for n in _to_name_list(e.get("System Name")):
            if n not in names:
                names.append(n)
        for r in e.get("Reference", []):
            if r not in refs:
                refs.append(r)
        for m in e.get("Detection Method", []):
            if m not in methods:
                methods.append(m)
        note = e.get("Notes")
        if note and note not in notes_bits:
            notes_bits.append(note)

    merged = {
        "System Name": names if len(names) > 1 else (names[0] if names else None),
        "RA": _pick_first_value(group_entries, "RA"),
        "Dec": _pick_first_value(group_entries, "Dec"),
        # Set Moltzer values to have top priority for Periods and eccentricity
        "Period": _pick_by_source_priority(group, "Period", ["Moltzer", "Oomen", "Kluska"], catalogs),
        "Eccentricity": _pick_by_source_priority(group, "Eccentricity", ["Moltzer", "Oomen", "Kluska"], catalogs),
        "M1": _pick_first_value(group_entries, "M1"),
        "M2": _pick_first_value(group_entries, "M2"),
        "Mass Function": _pick_first_value(group_entries, "Mass Function"),
        "M1_sin3i": _pick_first_value(group_entries, "M1_sin3i"),
        "M2_sin3i": _pick_first_value(group_entries, "M2_sin3i"),
        "evol_type_1": "MS",
        "evol_type_2": "AGB",
        "obs_type_1": "MS",
        "obs_type_2": "Post-AGB",
        "system_class": "Post-AGB binary",
        "Detection Method": methods if methods else ["RV"],
        "Reference": refs,
        "Notes": "; ".join(notes_bits) if notes_bits else None,
        "Simbad": None,
    }
    merged_entries.append(_finalize_system_name_and_notes(merged))
    included_keys.update(group)

# Add systems not part of any cross-catalog merge group.
for key in all_keys:
    if key in included_keys:
        continue
    src, idx = key
    merged_entries.append(_finalize_system_name_and_notes(dict(catalogs[src][idx])))

print(f"Built merged catalog with {len(merged_entries)} systems")

Built merged catalog with 87 systems


In [31]:
merged_null_name = [e for e in merged_entries if not e.get("System Name")]
merged_null_ra = [e for e in merged_entries if e.get("RA", [None, None, None])[1] is None]
merged_null_dec = [e for e in merged_entries if e.get("Dec", [None, None, None])[1] is None]

print(f"Merged QA -> total={len(merged_entries)}")
print(f"null System Name: {len(merged_null_name)}")
print(f"null RA: {len(merged_null_ra)}")
print(f"null Dec: {len(merged_null_dec)}")

if merged_null_name:
    print("Example null-name merged record:")
    print(json.dumps(merged_null_name[0], indent=2))

Merged QA -> total=87
null System Name: 0
null RA: 2
null Dec: 2


---
## 6. Merge into a single combined post AGB fiel

Merge all three sources into one file (`postAGB_combined.raw.json`).

**Field priority for overlapping systems:**
| Field | Priority |
|---|---|
| RA / Dec | Oomen (SIMBAD) -> Kluska (table coords) -> Moltzer (none) |
| Period / Eccentricity | Moltzer -> Oomen (symmetric ±) -> Kluska |
| Mass Function / M1 / M2 | Oomen only |
| System Name | union of all name variants |
| Reference | union of all bibcodes |
| Detection Method | prefer RV over Photometric |
| Notes | concatenated |

In [32]:
COMBINED_OUTPUT = RAW_JSON_DIR / "postAGB_combined.raw.json"

# --- Save as JSON array (compatible with Combine_and_process_data.py) ---
COMBINED_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with open(COMBINED_OUTPUT, "w", encoding="utf-8") as fh:
    fh.write("[\n")
    for i, system in enumerate(merged_entries):
        line = json.dumps(system, separators=(",", ": "), ensure_ascii=False)
        fh.write("  " + line)
        if i < len(merged_entries) - 1:
            fh.write(",\n")
        else:
            fh.write("\n")
    fh.write("]\n")

print(f"Saved {len(merged_entries)} entries to {COMBINED_OUTPUT}")

# --- Summary ---
n_duplicates = sum(1 for g in groups_map.values() if len(g) > 1)
n_unique   = len(merged_entries)
print(f"\n{n_duplicates} duplicate/overlap groups merged, {n_unique} total unique systems in combined catalog.")
print(f"\nFirst entry:\n{json.dumps(merged_entries[0], indent=2)}")

Saved 87 entries to /Users/liekevanson/Documents/Projects/post_mt_review/data/result_tables/raw_json/postAGB_combined.raw.json

53 duplicate/overlap groups merged, 87 total unique systems in combined catalog.

First entry:
{
  "System Name": "IRAS 09538-7622",
  "RA": [
    null,
    148.49374999999998,
    null
  ],
  "Dec": [
    null,
    -76.61472222222221,
    null
  ],
  "Period": [
    300.0,
    1190.0,
    300.0
  ],
  "Eccentricity": [
    null,
    null,
    null
  ],
  "M1": [
    null,
    null,
    null
  ],
  "M2": [
    null,
    null,
    null
  ],
  "Mass Function": [
    null,
    null,
    null
  ],
  "M1_sin3i": [
    null,
    null,
    null
  ],
  "M2_sin3i": [
    null,
    null,
    null
  ],
  "evol_type_1": "MS",
  "evol_type_2": "AGB",
  "obs_type_1": "MS",
  "obs_type_2": "Post-AGB",
  "system_class": "Post-AGB binary",
  "Detection Method": [
    "SED",
    "Photometric"
  ],
  "Reference": [
    "2022A&A...658A..36K",
    "Kiss2007"
  ],
  "Notes": "Alter

In [33]:
# Print number of entries with nonzero eccentricity
n_ecc = sum(1 for e in merged_entries
            if has_value(e.get("Eccentricity")) and float(e["Eccentricity"][1]) > 0)
print(f"\nNumber of systems with nonzero eccentricity: {n_ecc} / {n_unique} ({n_ecc/n_unique:.1%})")    

# Print number of systems with P value <= 0
n_period = sum(1 for e in merged_entries
               if has_value(e.get("Period")) and float(e["Period"][1]) <= 0)
print(f"\nNumber of systems with negative period measurement: {n_period} / {n_unique } ({n_period/n_unique:.1%})")   


Number of systems with nonzero eccentricity: 30 / 87 (34.5%)

Number of systems with negative period measurement: 0 / 87 (0.0%)


In [34]:
print("DEBUG: Check HD131356 specifically in name_to_keys:")
hd_keys = {k: v for k, v in name_to_keys.items() if '131356' in k}
for k in sorted(hd_keys.keys()):
    print(f"  '{k}': {hd_keys[k]}")

print("\nDEBUG: Groups with >1 entry (duplicate merges):")
dup_groups = [g for g in groups_map.values() if len(g) > 1]
print(f"  Total duplicate groups: {len(dup_groups)}")
for i, g in enumerate(dup_groups[:10]):
    catalogs_set = set(src for src, _ in g)
    print(f"  Group {i}: {g}, catalogs={catalogs_set}")

print(f"\nDEBUG: Multi_groups (≥2 catalogs): {len(multi_groups)}")
print(f"  merged_entries count: {len(merged_entries)}")




DEBUG: Check HD131356 specifically in name_to_keys:
  'ENTRAHD131356': [('Kluska', 23)]
  'HD131356': [('Oomen', 11), ('Moltzer', 32)]

DEBUG: Groups with >1 entry (duplicate merges):
  Total duplicate groups: 53
  Group 0: [('Kluska', 0), ('Oomen', 3), ('Moltzer', 4)], catalogs={'Moltzer', 'Oomen', 'Kluska'}
  Group 1: [('Kluska', 1), ('Oomen', 31), ('Moltzer', 22)], catalogs={'Moltzer', 'Oomen', 'Kluska'}
  Group 2: [('Kluska', 2), ('Moltzer', 28)], catalogs={'Moltzer', 'Kluska'}
  Group 3: [('Kluska', 6), ('Oomen', 8), ('Moltzer', 30)], catalogs={'Moltzer', 'Oomen', 'Kluska'}
  Group 4: [('Kluska', 7), ('Oomen', 28), ('Moltzer', 1)], catalogs={'Moltzer', 'Oomen', 'Kluska'}
  Group 5: [('Kluska', 8), ('Moltzer', 38)], catalogs={'Moltzer', 'Kluska'}
  Group 6: [('Kluska', 9), ('Oomen', 19), ('Moltzer', 15)], catalogs={'Moltzer', 'Oomen', 'Kluska'}
  Group 7: [('Kluska', 10), ('Moltzer', 39)], catalogs={'Moltzer', 'Kluska'}
  Group 8: [('Kluska', 11), ('Oomen', 20), ('Moltzer', 34)], c